In [29]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset

In [30]:
IMAGE_FOLDER = "Filtered_Train"

annotations = pd.read_csv("Filtered_Train/filtered_annotations.csv")

annotations.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,848617_jpg.rf.b6167489414b68fcfe2ccda38d24ac4a...,500.0,333.0,ahaetulla prasina,127.0,0.0,375.0,333.0
1,203459385_jpg.rf.73858ce8179c66e61c2203f4966e8...,500.0,375.0,bungarus caeruleus,144.0,48.0,332.0,325.0
2,178020213_jpg.rf.902a97ff37bebf466e6800e389762...,281.0,500.0,ptyas korros,0.0,161.0,281.0,347.0
3,AUG81008965_jpeg.rf.f10a9b8c28dba23257d0ee0594...,374.0,500.0,bungarus caeruleus,142.0,134.0,361.0,436.0
4,32814798_jpeg.rf.10944af1cd359dd50fca3e4518592...,243.0,500.0,ahaetulla prasina,60.0,129.0,179.0,325.0


In [31]:
classes = sorted(annotations["class"].unique())

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(classes)
}

idx_to_class = {
    idx: cls
    for cls, idx in class_to_idx.items()
}



In [32]:
print(class_to_idx)

{'ahaetulla prasina': 0, 'amphiesma stolatum': 1, 'bungarus caeruleus': 2, 'bungarus fasciatus': 3, 'chrysopelea ornata': 4, 'daboia russelii': 5, 'dendrelaphis pictus': 6, 'naja naja': 7, 'ophiophagus hannah': 8, 'psammodynastes pulverulentus': 9, 'ptyas korros': 10, 'ptyas mucosa': 11, 'trimeresurus albolabris': 12, 'trimeresurus purpureomaculatus': 13, 'xenochrophis piscator': 14}


In [33]:
annotations["Label"] = annotations["class"].map(class_to_idx)

annotations.sample(10)

,filename,width,height,class,xmin,ymin,xmax,ymax,Label
3609,AUG107288826_jpeg.rf.e09f742369d055431b8ecc0f1...,375.0,500.0,naja naja,47.0,6.0,267.0,488.0,7
556,162056358_jpeg.rf.ccf1abd0a5ebe0d22ab833a0d0a8...,500.0,364.0,psammodynastes pulverulentus,0.0,69.0,450.0,291.0,9
3756,110498956_jpeg.rf.70716c4c2c2e1cfcad4a13bb3565...,500.0,375.0,naja naja,193.0,59.0,308.0,285.0,7
2195,194241837_jpeg.rf.cb16f81b0e106b8cbc5102f34150...,500.0,302.0,ophiophagus hannah,79.0,108.0,315.0,248.0,8
317,45951976_jpeg.rf.6e6344e48f2a749494370c5201bea...,500.0,375.0,ahaetulla prasina,75.0,0.0,285.0,303.0,0
2679,213217602_jpg.rf.010b986f5d995eb40a8bd9efee4e4...,375.0,500.0,psammodynastes pulverulentus,0.0,178.0,307.0,276.0,9
965,103139424_jpg.rf.93750f52261c50f8cd2ebffd43692...,500.0,333.0,trimeresurus albolabris,2.0,47.0,308.0,213.0,12
5029,85104258_jpeg.rf.50384a4765710e726abf2d92444ea...,375.0,500.0,bungarus caeruleus,148.0,12.0,252.0,484.0,2
3480,651258_JPG.rf.ad11660b6b150f67d0e10ef3e817ef54...,500.0,375.0,ahaetulla prasina,29.0,42.0,262.0,293.0,0
2240,84202355_jpeg.rf.23bf55f169798760583448c554061...,500.0,375.0,dendrelaphis pictus,90.0,75.0,458.0,375.0,6


In [34]:
IMAGE_FOLDER = "Filtered_Train"
csv_file = "Filtered_Train/filtered_annotations.csv"

annotations = pd.read_csv(csv_file)

classes = sorted(annotations["class"].dropna().astype(str).unique())
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

annotations["Label"] = annotations["class"].astype(str).map(class_to_idx)

# Save a new CSV that contains the Label column
annotations.to_csv("Filtered_Train/filtered_annotations_with_label.csv", index=False)

annotations.head()

,filename,width,height,class,xmin,ymin,xmax,ymax,Label
0,848617_jpg.rf.b6167489414b68fcfe2ccda38d24ac4a...,500.0,333.0,ahaetulla prasina,127.0,0.0,375.0,333.0,0
1,203459385_jpg.rf.73858ce8179c66e61c2203f4966e8...,500.0,375.0,bungarus caeruleus,144.0,48.0,332.0,325.0,2
2,178020213_jpg.rf.902a97ff37bebf466e6800e389762...,281.0,500.0,ptyas korros,0.0,161.0,281.0,347.0,10
3,AUG81008965_jpeg.rf.f10a9b8c28dba23257d0ee0594...,374.0,500.0,bungarus caeruleus,142.0,134.0,361.0,436.0,2
4,32814798_jpeg.rf.10944af1cd359dd50fca3e4518592...,243.0,500.0,ahaetulla prasina,60.0,129.0,179.0,325.0,0


In [35]:
class SnakeDataset(Dataset):
    def __init__(self, csv_file, image_folder, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_folder = image_folder
        self.transform = transform

        # If Label is missing, create it from class
        if "Label" not in self.data.columns:
            if "class" not in self.data.columns:
                raise KeyError("CSV must contain 'class' or 'Label' column.")

            classes = sorted(self.data["class"].dropna().astype(str).unique())
            self.class_to_idx = {
                cls: idx for idx, cls in enumerate(classes)
            }

            self.data["Label"] = self.data["class"].astype(str).map(self.class_to_idx)
        else:
            self.class_to_idx = None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        image_path = os.path.join(self.image_folder, row["filename"])
        image = Image.open(image_path).convert("RGB")

        label = int(row["Label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [36]:
dataset = SnakeDataset(
    csv_file="Filtered_Train/filtered_annotations_with_label.csv",
    image_folder="Filtered_Train"
)

print(len(dataset))
image, label = dataset[0]

print(type(image))
print(image.size)
print(label)

6110
<class 'PIL.Image.Image'>
(500, 333)
0


In [37]:
print(len(dataset))

6110


In [38]:
image, label = dataset[0]

print(type(image))
print(image.size)
print(label)

<class 'PIL.Image.Image'>
(500, 333)
0


In [39]:
from torchvision import transforms

In [40]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [41]:
dataset = SnakeDataset(
    csv_file="Filtered_Train/filtered_annotations.csv",
    image_folder="Filtered_Train",
    transform=train_transform
)

In [42]:
image, label = dataset[0]

print(type(image))
print(image.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
0


In [43]:
DataLoader = torch.utils.data.DataLoader
train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

In [44]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels)

torch.Size([4, 3, 224, 224])
tensor([1, 5, 6, 6])


In [45]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [46]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i, ax in enumerate(axes.flat):

    img = images[i].permute(1,2,0).numpy()

    ax.imshow(img)
    ax.set_title(idx_to_class[labels[i].item()])
    ax.axis("off")

plt.tight_layout()

plt.savefig("reports/sample_batch.png", dpi=200)

print("Saved Successfully")

Saved Successfully
